# GloGEM test: Aletsch & Morteratsch — output visualisation

This notebook reads and visualises GloGEM outputs from the six test runs.
Run the model first (from the GloGEM root in IDL):

```
cp test/configs/config_aletsch_daily_calib.pro        config.pro  &&  .r glogem
cp test/configs/config_aletsch_daily_hindcast.pro     config.pro  &&  .r glogem
cp test/configs/config_aletsch_daily_projection.pro   config.pro  &&  .r glogem
cp test/configs/config_aletsch_monthly_calib.pro      config.pro  &&  .r glogem
cp test/configs/config_aletsch_monthly_hindcast.pro   config.pro  &&  .r glogem
cp test/configs/config_aletsch_monthly_projection.pro config.pro  &&  .r glogem
```

Then open this notebook (in Jupyter or VS Code) and run all cells. Requires
only `numpy` and `matplotlib`. The data-reading and plotting code lives in
`glogem_plots.py` next to this notebook, so this file stays focused on the
walk-through rather than matplotlib boilerplate.

## 1. Setup

Paths are detected automatically from the notebook's working directory —
no manual editing required (open/run this notebook from the `test/` folder).

In [ ]:
from pathlib import Path
import numpy as np
import glogem_plots as gp

gp.setup_style()

# Locate the test directory and derive all paths
TEST_DIR = Path.cwd()
OUT_DIR = TEST_DIR / 'outputs'
CC = '_Aletsch_Morteratsch'   # catchment suffix in output filenames
PLOT_DIR = TEST_DIR / 'example_plots'
PLOT_DIR.mkdir(exist_ok=True)

required_files = [
    OUT_DIR / f'daily/CentralEurope/PAST/PAST_original/centraleurope_Annual_Balance_sfc_r1{CC}.dat',
    OUT_DIR / f'monthly/CentralEurope/PAST/PAST_original/centraleurope_Annual_Balance_sfc_r1{CC}.dat',
    OUT_DIR / f'daily/CentralEurope/files/files_original/MRI-ESM2-0/ssp126/centraleurope_Area_r1{CC}.dat',
    OUT_DIR / f'monthly/CentralEurope/files/files_original/MRI-ESM2-0/ssp126/centraleurope_Area_r1{CC}.dat',
    OUT_DIR / f'daily/CentralEurope/files/files_original/MRI-ESM2-0/ssp126/hypsometry.zip',
]
missing = [f for f in required_files if not f.exists()]
if missing:
    for f in missing:
        print('MISSING:', f)
    print('Run the model steps first (see notebook header).')
else:
    print('Output files found - ready to visualise.')

## 2. Calibration results

Per-glacier calibrated parameters from both the daily and monthly runs.

In [ ]:
cal_d = gp.read_calibration(OUT_DIR / f'daily/CentralEurope/calibration/calibrate_m1_cID9_centraleurope_final_era5{CC}.dat')
cal_m = gp.read_calibration(OUT_DIR / f'monthly/CentralEurope/calibration/calibrate_m1_cID9_centraleurope_final_era5{CC}.dat')

idx_al_c  = int(np.where(cal_d['id'] == '02596')[0][0])
idx_mo_c  = int(np.where(cal_d['id'] == '02216')[0][0])
idx_al_cm = int(np.where(cal_m['id'] == '02596')[0][0])
idx_mo_cm = int(np.where(cal_m['id'] == '02216')[0][0])

print('Daily calibration:')
print(f"  Aletsch:     Ba={cal_d['ba'][idx_al_c]:6.3f}  ELA={cal_d['ela'][idx_al_c]:5.0f}  AAR={cal_d['aar'][idx_al_c]:5.1f}  DDFsnow={cal_d['ddfs'][idx_al_c]:.2f}  DDFice={cal_d['ddfi'][idx_al_c]:.2f}  c_prec={cal_d['cprec'][idx_al_c]:.2f}")
print(f"  Morteratsch: Ba={cal_d['ba'][idx_mo_c]:6.3f}  ELA={cal_d['ela'][idx_mo_c]:5.0f}  AAR={cal_d['aar'][idx_mo_c]:5.1f}  DDFsnow={cal_d['ddfs'][idx_mo_c]:.2f}  DDFice={cal_d['ddfi'][idx_mo_c]:.2f}  c_prec={cal_d['cprec'][idx_mo_c]:.2f}")

print()
print('Monthly calibration:')
print(f"  Aletsch:     Ba={cal_m['ba'][idx_al_cm]:6.3f}  DDFsnow={cal_m['ddfs'][idx_al_cm]:.2f}  DDFice={cal_m['ddfi'][idx_al_cm]:.2f}  c_prec={cal_m['cprec'][idx_al_cm]:.2f}")
print(f"  Morteratsch: Ba={cal_m['ba'][idx_mo_cm]:6.3f}  DDFsnow={cal_m['ddfs'][idx_mo_cm]:.2f}  DDFice={cal_m['ddfi'][idx_mo_cm]:.2f}  c_prec={cal_m['cprec'][idx_mo_cm]:.2f}")

## 3. Read hindcast time series

In [ ]:
# ---- Daily hindcast ----
gl_ids_d, yrs_d, mb_d = gp.read_series(OUT_DIR / f'daily/CentralEurope/PAST/PAST_original/centraleurope_Annual_Balance_sfc_r1{CC}.dat')
idx_al = int(np.where(gl_ids_d == '02596')[0][0])
idx_mo = int(np.where(gl_ids_d == '02216')[0][0])

# ---- Monthly hindcast ----
gl_ids_m, yrs_m, mb_m = gp.read_series(OUT_DIR / f'monthly/CentralEurope/PAST/PAST_original/centraleurope_Annual_Balance_sfc_r1{CC}.dat')
idx_al_m = int(np.where(gl_ids_m == '02596')[0][0])
idx_mo_m = int(np.where(gl_ids_m == '02216')[0][0])

# ---- ELA and AAR (daily) ----
_, _, ela_d = gp.read_series(OUT_DIR / f'daily/CentralEurope/PAST/PAST_original/centraleurope_ELA_r1{CC}.dat')
_, _, aar_d = gp.read_series(OUT_DIR / f'daily/CentralEurope/PAST/PAST_original/centraleurope_AAR_r1{CC}.dat')

print('Data loaded.')
print(f"  Daily   mean Ba - Aletsch: {mb_d[idx_al].mean():6.3f}   Morteratsch: {mb_d[idx_mo].mean():6.3f} m w.e./yr")
print(f"  Monthly mean Ba - Aletsch: {mb_m[idx_al_m].mean():6.3f}   Morteratsch: {mb_m[idx_mo_m].mean():6.3f} m w.e./yr")

## 4. Plots

In [ ]:
# ---- Plot 1: Daily annual mass balance ----
gp.plot_annual_mb(yrs_d, mb_d, idx_al, idx_mo, PLOT_DIR);

In [ ]:
# ---- Plot 2: ELA and AAR ----
gp.plot_ela_aar(yrs_d, ela_d, aar_d, idx_al, PLOT_DIR);

In [ ]:
# ---- Plot 3: Daily vs monthly comparison ----
gp.plot_daily_vs_monthly(yrs_d, mb_d, yrs_m, mb_m, idx_al, idx_mo, idx_al_m, idx_mo_m, PLOT_DIR);

In [ ]:
# ---- Plot 4: Calibrated parameters ----
# Al-d=Aletsch daily, Al-m=Aletsch monthly, Mo-d=Morteratsch daily, Mo-m=Morteratsch monthly
gp.plot_calibrated_params(cal_d, cal_m, idx_al_c, idx_mo_c, idx_al_cm, idx_mo_cm, PLOT_DIR);

## 5. Projection results (to 2100)

The projection runs (Steps 5 & 6) use GMIP4 / MRI-ESM2-0 / ssp126 with
`glacier_retreat='y'`, so glacier area and volume are expected to decline
substantially over the 21st century. 2x2 mosaic: rows are Area/Volume,
columns are Aletsch (left) / Morteratsch (right).

In [ ]:
# ---- Read projected Area and Volume (daily and monthly) ----
# Index lookups use each file's own id array rather than reusing the
# hindcast indices, since projection output is a separate set of files.
ids_pd, yrs_proj_d, area_d = gp.read_series(OUT_DIR / f'daily/CentralEurope/files/files_original/MRI-ESM2-0/ssp126/centraleurope_Area_r1{CC}.dat')
ids_pm, yrs_proj_m, area_m = gp.read_series(OUT_DIR / f'monthly/CentralEurope/files/files_original/MRI-ESM2-0/ssp126/centraleurope_Area_r1{CC}.dat')
_, _, vol_d = gp.read_series(OUT_DIR / f'daily/CentralEurope/files/files_original/MRI-ESM2-0/ssp126/centraleurope_Volume_r1{CC}.dat')
_, _, vol_m = gp.read_series(OUT_DIR / f'monthly/CentralEurope/files/files_original/MRI-ESM2-0/ssp126/centraleurope_Volume_r1{CC}.dat')

idx_al_pd = int(np.where(ids_pd == '02596')[0][0])
idx_mo_pd = int(np.where(ids_pd == '02216')[0][0])
idx_al_pm = int(np.where(ids_pm == '02596')[0][0])
idx_mo_pm = int(np.where(ids_pm == '02216')[0][0])

gp.plot_projection_mosaic(yrs_proj_d, area_d, vol_d, yrs_proj_m, area_m, vol_m,
                           idx_al_pd, idx_mo_pd, idx_al_pm, idx_mo_pm, PLOT_DIR)

print(f"Aletsch:     area {area_d[idx_al_pd, 0]:.1f}->{area_d[idx_al_pd, -1]:.1f} km2, volume {vol_d[idx_al_pd, 0]:.2f}->{vol_d[idx_al_pd, -1]:.2f} km3 (1991->2099, daily)")
print(f"Morteratsch: area {area_d[idx_mo_pd, 0]:.1f}->{area_d[idx_mo_pd, -1]:.1f} km2, volume {vol_d[idx_mo_pd, 0]:.2f}->{vol_d[idx_mo_pd, -1]:.2f} km3 (1991->2099, daily)")

## 6. Glacier elevation profile

Bedrock and glacier surface elevation along each glacier, from the terminus
(0 km) to the highest elevation band. Requires `write_hypsometry_files='y'`
in the projection config (already set in `config_aletsch_daily_projection.pro`),
which writes per-decade area and volume per elevation band into a
`hypsometry.zip` archive next to the other projection output.

```{note}
GloGEM's standard (non-`long_GCM`) forward runs hard-clamp `tran[1]` to 2100,
and `tran[1]` itself is *exclusive* (`years = tran[1] - tran[0]`), so the last
calendar year actually simulated is 2099 and the last full decade snapshot
captured is 2090. The plot below labels this final line "2100" as shorthand
for "end of the projected century" — the underlying data point is 2090.
```

In [ ]:
hypso_zip = OUT_DIR / f'daily/CentralEurope/files/files_original/MRI-ESM2-0/ssp126/hypsometry.zip'
geo_al, dec_years, thick_decade_al = gp.glacier_profile(
    '02596', TEST_DIR / 'geometricdata/rgiv7/bands/centraleurope/02596.dat', hypso_zip)
geo_mo, _, thick_decade_mo = gp.glacier_profile(
    '02216', TEST_DIR / 'geometricdata/rgiv7/bands/centraleurope/02216.dat', hypso_zip)

gp.plot_glacier_profile(geo_al, geo_mo, dec_years, thick_decade_al, thick_decade_mo, PLOT_DIR);

## 7. Sanity checks

Calibrated Ba within +/-0.3 m w.e./yr of the geodetic target, DDFs within
plausible physical bounds, and substantial projected area loss by 2100
indicate the model is working correctly.

In [ ]:
print()
print('====== Sanity checks ======')
n_pass = 0
n_total = 0

def check(label, value, lo, hi):
    global n_pass, n_total
    n_total += 1
    ok = lo <= value <= hi
    if ok:
        n_pass += 1
        print(f'  PASS  {label}: {value:7.3f}')
    else:
        print(f'  FAIL  {label}: {value:7.3f}  [expected {lo:.2f} to {hi:.2f}]')

check('Aletsch   daily  Ba',    cal_d['ba'][idx_al_c],  -1.5, -0.8)
check('Morteratsch daily  Ba',  cal_d['ba'][idx_mo_c],  -1.4, -0.5)
check('Aletsch   monthly Ba',   cal_m['ba'][idx_al_cm], -1.5, -0.8)
check('Morteratsch monthly Ba', cal_m['ba'][idx_mo_cm], -1.4, -0.5)

check('Aletsch daily DDFsnow', cal_d['ddfs'][idx_al_c], 1.5, 7.5)
check('Aletsch daily DDFice',  cal_d['ddfi'][idx_al_c], 3.0, 15.0)

ii_d = (yrs_d >= 2000) & (yrs_d <= 2020)
ii_m = (yrs_m >= 2000) & (yrs_m <= 2020)
check('Aletsch   daily  mean Ba 2000-2020',   mb_d[idx_al, ii_d].mean(),  -1.5, -0.8)
check('Morteratsch daily  mean Ba 2000-2020', mb_d[idx_mo, ii_d].mean(),  -1.4, -0.5)
check('Aletsch   monthly mean Ba 2000-2020',  mb_m[idx_al_m, ii_m].mean(), -1.5, -0.8)
check('Morteratsch monthly mean Ba 2000-2020',mb_m[idx_mo_m, ii_m].mean(), -1.4, -0.5)

check('Aletsch daily mean ELA 1991-2020', ela_d[idx_al].mean(), 2600, 3300)

# Projection retreat checks — area in 2100 should be well below 1991
check('Aletsch     2100 area loss (%, daily)',   100 * (1 - area_d[idx_al_pd, -1] / area_d[idx_al_pd, 0]),  15, 90)
check('Morteratsch 2100 area loss (%, daily)',   100 * (1 - area_d[idx_mo_pd, -1] / area_d[idx_mo_pd, 0]),  15, 90)

print()
print(f'{n_pass} / {n_total} checks passed')
if n_pass == n_total:
    print('All checks passed - the model is working correctly!')
else:
    print('Some checks failed - inspect the output files for details.')

## 8. Next steps

- **Your own region**: copy any test config to `config.pro`, change
  `catchment_selection`, `region_id_loop`, `dirres`, and the data paths
  to your production locations.
- **More GCMs / SSPs**: the projection configs use a single GMIP4
  model/scenario (MRI-ESM2-0 / ssp126) matching the bundled test data.
  For a full ensemble, set `MIP`, `GCM_model_idx`, `GCM_rcp_idx` per the
  [GCM configuration docs](../running-glogem/gcm-configuration.md) and
  point `dir_clim` at a full climate data share.
- **More physics**: uncomment `firnice_temperature='y'` or
  `debris_supraglacial='y'` in the config.